# Custom Model Backends

This notebook explores how to configure different model backends in Anvil, add custom models, use cost optimization routing, and compare outputs across models.

## 1. Setting Up Different Model Backends

In [ ]:
from anvil.engine import AnvilEngine, AnvilConfig
from anvil.models import LocalModel, OpenAIModel, AnthropicModel
from rich.console import Console
from rich.table import Table

console = Console()

# --- Local Model (Ollama) ---
local_engine = AnvilEngine(
    model_backend="local",
    model_name="fableforge-14b",
    workspace=".",
)
console.print("[green]✓[/green] Local engine created (Ollama + fableforge-14b)")

# --- OpenAI ---
# Requires: export OPENAI_API_KEY=sk-...
openai_engine = AnvilEngine(
    model_backend="openai",
    model_name="gpt-4o",
    workspace=".",
)
console.print("[green]✓[/green] OpenAI engine created (gpt-4o)")

# --- Anthropic ---
# Requires: export ANTHROPIC_API_KEY=sk-ant-...
anthropic_engine = AnvilEngine(
    model_backend="anthropic",
    model_name="claude-sonnet-4-20250514",
    workspace=".",
)
console.print("[green]✓[/green] Anthropic engine created (claude-sonnet-4)")

## 2. Configuring via Config File

In [ ]:
# Create a config file for persistent settings
config_yaml = """model:
  backend: openai
  name: gpt-4o
  api_key_env: OPENAI_API_KEY
  temperature: 0.7
  max_tokens: 4096

engine:
  max_iterations: 15
  max_retries: 3
  verify: true

verify:
  strictness: balanced
  checkers:
    - syntax
    - tests
    - lint
    - diff_review

daemon:
  host: 127.0.0.1
  port: 8420
"""

with open("anvil.config.yaml", "w") as f:
    f.write(config_yaml)

config = AnvilConfig.from_file("anvil.config.yaml")
engine = AnvilEngine(config=config)
console.print("[green]✓[/green] Engine created from config file")
console.print(f"  Backend: {config.model_backend}")
console.print(f"  Model: {config.model_name}")
console.print(f"  Verify: {config.verify}")

## 3. Adding a Custom Model Backend

In [ ]:
from anvil.models import BaseModel, Message, ModelResponse
from dataclasses import dataclass
import time

class VLLMModel(BaseModel):
    """Custom model backend for vLLM-hosted models."""
    
    def __init__(self, model_name: str, base_url: str = "http://localhost:8000", **kwargs):
        super().__init__(model_name=model_name, **kwargs)
        self.base_url = base_url.rstrip("/")
        self._client = None
    
    def _get_client(self):
        """Lazy import and create the OpenAI client (vLLM is OpenAI-compatible)."""
        if self._client is None:
            from openai import OpenAI
            self._client = OpenAI(
                base_url=f"{self.base_url}/v1",
                api_key="not-needed",  # vLLM doesn't require auth by default
            )
        return self._client
    
    def generate(self, messages: list[Message], **kwargs) -> ModelResponse:
        """Generate a response using the vLLM model."""
        client = self._get_client()
        start_time = time.time()
        
        # Convert Anvil messages to OpenAI format
        api_messages = [
            {"role": m.role, "content": m.content}
            for m in messages
        ]
        
        response = client.chat.completions.create(
            model=self.model_name,
            messages=api_messages,
            temperature=kwargs.get("temperature", 0.7),
            max_tokens=kwargs.get("max_tokens", 4096),
        )
        
        return ModelResponse(
            content=response.choices[0].message.content,
            model=response.model,
            usage=response.usage,
            finish_reason=response.choices[0].finish_reason,
            tool_calls=None,
            raw=response.model_dump(),
        )
    
    def count_tokens(self, text: str) -> int:
        """Estimate token count (rough approximation)."""
        return len(text) // 4  # Rough: ~4 chars per token
    
    def validate_connection(self) -> bool:
        """Test connection to vLLM server."""
        import requests
        try:
            resp = requests.get(f"{self.base_url}/health", timeout=5)
            return resp.status_code == 200
        except Exception:
            return False

# Create engine with custom model
vllm_model = VLLMModel(
    model_name="fableforge-14b",
    base_url="http://localhost:8000",
)

# Use the custom model
custom_config = AnvilConfig(model_backend="custom")
engine = AnvilEngine(config=custom_config)
engine.model = vllm_model
console.print("[green]✓[/green] Custom vLLM model backend created")
console.print(f"  Connection valid: {vllm_model.validate_connection()}")

## 4. Using Local Models (Ollama)

In [ ]:
# Connect to Ollama running locally
from anvil.models import LocalModel

# Default: Ollama on localhost:11434
local_model = LocalModel(
    model_name="fableforge-14b",
    base_url="http://localhost:11434",
    timeout=120,
    context_length=8192,
)

# For vLLM instead of Ollama
vllm_local = LocalModel(
    model_name="fableforge-14b",
    base_url="http://localhost:8000",
    api_type="vllm",  # Use vLLM API format
    context_length=8192,
)

# Test the model directly
from anvil.models import Message

messages = [
    Message(role="system", content="You are a Python coding expert."),
    Message(role="user", content="Write a function to check if a string is a palindrome."),
]

# response = local_model.generate(messages)
# print(response.content)
console.print("[dim]Uncomment the lines above to test local model generation.[/dim]")

## 5. Cost Optimization Routing

In [ ]:
from anvil.integrations import CostOptimizerBridge

# Set up cost optimization with budget tracking
optimizer = CostOptimizerBridge(
    default_model="fableforge-14b",  # Fallback to local model
    budget_usd=1.0,                  # $1.00 total budget
    strategy="best_value",           # Balance cost and quality
)

# Attach to engine — all run() calls are now cost-optimized
engine = AnvilEngine(model_backend="local")
optimized_engine = optimizer.attach(engine)

console.print("[green]✓[/green] Cost optimizer attached")
console.print(f"  Strategy: {optimizer.strategy}")
console.print(f"  Budget: ${optimizer.budget_usd:.2f}")
console.print(f"  Default model: {optimizer.default_model}")

In [ ]:
# Run tasks with cost optimization
# The optimizer automatically routes to the cheapest adequate model
task = "Add a docstring to the calculator.py file"

# result = optimized_engine.run(task)
# console.print(f"Result: {result.summary}")
# console.print(f"Cost: ${result.cost.total_usd:.4f}")
# console.print(f"Model used: {result.cost.model_used}")

# Compare strategies
console.print("\n[bold]Cost Optimization Strategies[/bold]\n")
table = Table()
table.add_column("Strategy", style="cyan")
table.add_column("Behavior")
table.add_column("Best For")
table.add_column("Cost", style="green")

table.add_row("cheapest", "Always use cheapest model", "Prototyping, testing", "Low")
table.add_row("best_value", "Balance cost and quality", "Production", "Medium")
table.add_row("quality_first", "Always use best model", "Critical tasks", "High")

console.print(table)

## 6. Comparing Model Outputs

In [ ]:
# Run the same task across multiple models and compare
import concurrent.futures

task = "Write a Python function that finds the longest palindromic substring in a given string."

models = {
    "FableForge-14B (local)": AnvilEngine(model_backend="local", model_name="fableforge-14b"),
    "GPT-4o (OpenAI)": AnvilEngine(model_backend="openai", model_name="gpt-4o"),
    "Claude Sonnet (Anthropic)": AnvilEngine(model_backend="anthropic", model_name="claude-sonnet-4-20250514"),
}

def run_model(name_engine):
    name, engine = name_engine
    try:
        result = engine.plan(task)  # Plan only for speed
        return name, result.steps, None
    except Exception as e:
        return name, None, str(e)

# Run in parallel
# results = dict(concurrent.futures.ThreadPoolExecutor().map(run_model, models.items()))
# for name, steps, error in results:
#     if error:
#         console.print(f"[red]{name}: Error - {error}[/red]")
#     else:
#         console.print(f"[green]{name}[/green]: {len(steps)} steps")
#         for step in steps:
#             console.print(f"  {step.id}. {step.description}")

console.print("[dim]Uncomment the code above to run comparison (requires API keys).[/dim]")
console.print("\n[bold]Expected Output:[/bold]")
console.print("FableForge-14B (local): 4 steps")
console.print("GPT-4o (OpenAI): 3 steps")
console.print("Claude Sonnet (Anthropic): 4 steps")

## 7. Summary

You've learned:

1. **Multiple backends** — Local (Ollama/vLLM), OpenAI, Anthropic
2. **Config files** — Persistent YAML configuration
3. **Custom models** — Implement `BaseModel` for any API
4. **Cost optimization** — Route to the cheapest adequate model
5. **Comparison** — Run the same task across models

Next: Try the training pipeline notebook to train your own models!